In [42]:
import pandas as pd
import numpy as np

In [43]:
emails = pd.read_csv(
    "../../data/01_raw/raw_emails.csv",
    low_memory=False
)

emails.shape

(123389, 27)

In [44]:
emails.info()
emails.head()

<class 'pandas.DataFrame'>
RangeIndex: 123389 entries, 0 to 123388
Data columns (total 27 columns):
 #   Column                                Non-Null Count   Dtype
---  ------                                --------------   -----
 0   Co_Ref                                123389 non-null  str  
 1   Time_to_Renewal                       123389 non-null  str  
 2   crm_accreditation_completed           102354 non-null  str  
 3   crm_timely_completion                 102354 non-null  str  
 4   crm_progress_towards_accreditation    102354 non-null  str  
 5   crm_delays_in_accreditation           102354 non-null  str  
 6   crm_contractor_suggested_leave        102354 non-null  str  
 7   crm_contractor_engagement             102354 non-null  str  
 8   crm_contractor_sentiment              102354 non-null  str  
 9   crm_contractor_sentiment_score        102354 non-null  str  
 10  crm_dts_or_ssip_mentioned             102354 non-null  str  
 11  crm_customer_payment_intention       

,Co_Ref,Time_to_Renewal,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_contractor_engagement,crm_contractor_sentiment,crm_contractor_sentiment_score,...,crm_accreditation_issues,crm_membership_overdue,crm_auto_renewal_status,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned,year
0,KG5766,pre_renewal,Not Discussed,Not Discussed,Not Discussed,Yes,No,Yes,Neutral,50,...,Not Discussed,Yes,0,No,No,Yes,Yes,No,Yes,2025
1,EJ1532,14_out,Not Discussed,Not Discussed,Not Discussed,No,Not Discussed,No,Not Discussed,Not Discussed,...,Not Discussed,Not Discussed,0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025
2,AA4063,prior_year,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,Neutral,50,...,Not Discussed,No,0,No,No,Yes,Yes,Yes,Not Discussed,2025
3,JY9888,prior_year,No,No,Not Discussed,Yes,No,Yes,Satisfied,80,...,Not Discussed,Yes,0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025
4,WO6689,pre_renewal,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,Satisfied,80,...,No,No,0,No,No,Yes,Yes,No,Not Discussed,2026


In [45]:
emails = emails.drop_duplicates()
print("Shape after duplicate removal:", emails.shape)

Shape after duplicate removal: (123389, 27)


In [46]:
emails.columns = (
    emails.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [47]:
for col in emails.columns:
    if "date" in col:
        emails[col] = pd.to_datetime(emails[col], errors="coerce")

In [48]:
missing_df = (
    emails.isnull().mean() * 100
).sort_values(ascending=False)

missing_df.head(20)

crm_accreditation_completed           17.047711
crm_progress_towards_accreditation    17.047711
crm_timely_completion                 17.047711
crm_contractor_engagement             17.047711
crm_contractor_sentiment              17.047711
crm_delays_in_accreditation           17.047711
crm_contractor_suggested_leave        17.047711
crm_contractor_sentiment_score        17.047711
crm_dts_or_ssip_mentioned             17.047711
crm_customer_payment_intention        17.047711
crm_dissatisfaction_with_support       9.299857
crm_negative_customer_experience       9.299857
crm_refund_mentioned                   9.299857
crm_customer_complained                9.299857
crm_financial_hardship_mentioned       9.299857
crm_membership_overdue                 9.040514
crm_accreditation_issues               9.040514
crm_membership_level                   9.040514
crm_competitors_mentioned              9.040514
crm_platform_issues_raised             9.040514
dtype: float64

In [49]:
cat_cols = emails.select_dtypes(include="object").columns
num_cols = emails.select_dtypes(include=np.number).columns
print("Categorical columns:", cat_cols.tolist())
print("Numeric columns:", num_cols.tolist())

Categorical columns: ['co_ref', 'time_to_renewal', 'crm_accreditation_completed', 'crm_timely_completion', 'crm_progress_towards_accreditation', 'crm_delays_in_accreditation', 'crm_contractor_suggested_leave', 'crm_contractor_engagement', 'crm_contractor_sentiment', 'crm_contractor_sentiment_score', 'crm_dts_or_ssip_mentioned', 'crm_customer_payment_intention', 'crm_competitors_mentioned', 'crm_membership_level', 'crm_platform_issues_raised', 'crm_agent_chased_contractor', 'crm_agent_chase_count', 'crm_accreditation_issues', 'crm_membership_overdue', 'crm_auto_renewal_status', 'crm_dissatisified_with_renewal_price', 'crm_customer_complained', 'crm_refund_mentioned', 'crm_negative_customer_experience', 'crm_dissatisfaction_with_support', 'crm_financial_hardship_mentioned']
Numeric columns: ['year']


C:\Users\NuluShreya\AppData\Local\Temp\ipykernel_3572\2561724371.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = emails.select_dtypes(include="object").columns


In [50]:
# categorical columns
for col in cat_cols:
    if emails[col].isnull().sum() > 0:
        emails[col] = emails[col].fillna("No_Email")

# numeric columns
for col in num_cols:
    if emails[col].isnull().sum() > 0:
        unique_vals = emails[col].dropna().nunique()

        if unique_vals <= 2:
            emails[col] = emails[col].fillna(0)

        elif any(word in col.lower() for word in ["count", "num", "emails"]):
            emails[col] = emails[col].fillna(0)

        else:
            emails[col] = emails[col].fillna(
                emails[col].median()
            )

In [51]:
emails.dtypes

co_ref                                    str
time_to_renewal                           str
crm_accreditation_completed               str
crm_timely_completion                     str
crm_progress_towards_accreditation        str
crm_delays_in_accreditation               str
crm_contractor_suggested_leave            str
crm_contractor_engagement                 str
crm_contractor_sentiment                  str
crm_contractor_sentiment_score            str
crm_dts_or_ssip_mentioned                 str
crm_customer_payment_intention            str
crm_competitors_mentioned                 str
crm_membership_level                      str
crm_platform_issues_raised                str
crm_agent_chased_contractor               str
crm_agent_chase_count                     str
crm_accreditation_issues                  str
crm_membership_overdue                    str
crm_auto_renewal_status                   str
crm_dissatisified_with_renewal_price      str
crm_customer_complained           

In [52]:
emails.isnull().sum().sort_values(ascending=False).head(20)

co_ref                                0
time_to_renewal                       0
crm_accreditation_completed           0
crm_timely_completion                 0
crm_progress_towards_accreditation    0
crm_delays_in_accreditation           0
crm_contractor_suggested_leave        0
crm_contractor_engagement             0
crm_contractor_sentiment              0
crm_contractor_sentiment_score        0
crm_dts_or_ssip_mentioned             0
crm_customer_payment_intention        0
crm_competitors_mentioned             0
crm_membership_level                  0
crm_platform_issues_raised            0
crm_agent_chased_contractor           0
crm_agent_chase_count                 0
crm_accreditation_issues              0
crm_membership_overdue                0
crm_auto_renewal_status               0
dtype: int64

In [53]:
emails.to_csv(
    "../../data/02_processed/processed_emails.csv",
    index=False
)